# 03 · Baseline reproduction: Sec. 6 benchmark with the released DA3 weights
Released benchmark harness (frame sampling seed 42, max 100 frames, evo Umeyama + RANSAC, Open3D
TSDF, F1 / Chamfer) with **our** model plugged in. Rows: **[official]** = released weights in our
modules; **[reduced-scale]** = our checkpoint (attach the 02_train output, set `CKPT`).
Data: HF `depth-anything/DA3-BENCH` (HiRoom 0.7 GB, DTU-64 1.8 GB, 7Scenes 3.6 GB, DTU 8.9 GB,
ScanNet++ 10.8 GB, ETH3D 15.1 GB). Limited runs are labelled by `--max-scenes`.

In [ ]:
# --- repo setup (identical in every notebook) ---------------------------------------------
REPO_URL = "https://github.com/davideCabitz/DA3_LAB_Advanced_CV.git"   # push your branch first
BRANCH = "main"
import os, subprocess, sys
os.chdir("/kaggle/working")
url = REPO_URL
try:  # private repo: add a Kaggle secret named GITHUB_TOKEN (Add-ons > Secrets)
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.exists("DA3_LAB_Advanced_CV"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "DA3_LAB_Advanced_CV"], check=True)
else:
    subprocess.run(["git", "-C", "DA3_LAB_Advanced_CV", "pull", "--ff-only"], check=True)
os.chdir("/kaggle/working/DA3_LAB_Advanced_CV")
!pip install -q -r requirements.txt && pip install -q -e . --no-deps
sys.path.insert(0, os.path.abspath("src"))   # the editable install is only seen by new processes, not this kernel
!git log -1 --format="commit %H  %cd"

In [ ]:
# --- official DA3 repo (reference / parity oracle / benchmark harness), pinned commit ----------
import os, subprocess
if not os.path.exists("/kaggle/working/Depth-Anything-3"):
    subprocess.run(["git", "clone", "https://github.com/ByteDance-Seed/Depth-Anything-3.git",
                    "/kaggle/working/Depth-Anything-3"], check=True)
subprocess.run(["git", "-C", "/kaggle/working/Depth-Anything-3", "checkout", "3d835ec1a5802d64a8b8b15f817a1ab54809bfe4"], check=True)
OFFICIAL_SRC = "/kaggle/working/Depth-Anything-3/src"
!pip install -q -r requirements-eval.txt
# Open3D: requirements-eval installs the newest one; 0.20.0's TSDF returns empty meshes (isl-org/Open3D#7592),
# so use the verified 0.19.0 where it exists (Python <= 3.12)
import sys
if sys.version_info[:2] <= (3, 12):
    !pip install -q "open3d==0.19.0"
# Open3D self-test: does TSDF fusion (depth maps -> mesh) work in this environment?
!python scripts/check_open3d.py --out /kaggle/working/open3d_check.json

In [ ]:
import json
chk = json.load(open("/kaggle/working/open3d_check.json"))
RECON = bool(chk["legacy_tsdf_ok"])   # reconstruction scores (F1 / Chamfer) need a working Open3D TSDF
if RECON:
    print("Open3D TSDF OK:", chk["open3d"], "-> reconstruction scores (F1 / Chamfer) included")
else:
    print("Open3D", chk["open3d"], "on Python", chk["python"], "has a broken TSDF fusion (known Open3D 0.20 bug, "
          "github.com/isl-org/Open3D/issues/7592; the working 0.19.0 only exists up to Python 3.12).")
    print("This run SKIPS the reconstruction scores (F1 / Chamfer). Pose accuracy (AUC), failure detection "
          "and depth uncertainty are not affected.")
MODES = "pose recon_unposed" if RECON else "pose"
RECON_VARIANTS = "" if RECON else "--recon-variants"   # an empty list = pose-only harness replays
MODES = "pose recon_unposed recon_posed" if RECON else "pose"   # this notebook also scores recon_posed

In [ ]:
# --- reconstruction scores: Python 3.12 side environment (only when the Open3D above is broken) --------
# Kaggle / Colab run Python 3.13, whose only Open3D (0.20.0) has a broken TSDF fusion (isl-org/Open3D#7592),
# and the working 0.19.0 needs Python <= 3.12. The scoring (CPU only: TSDF fusion, F1 / Chamfer, pose AUC)
# then runs in a small Python 3.12 environment; DA3 itself still runs on the GPU in this notebook's Python.
import json, os
PY312 = "/tmp/py312/bin/python"
SPLIT = False   # True = model in this Python (--infer-only), scores in PY312 (--eval-only)
if not RECON:
    !pip install -q uv
    !uv python install 3.12
    if not os.path.exists(PY312):
        !uv venv /tmp/py312 --python 3.12
    !uv pip install -q --python {PY312} torch torchvision --index-url https://download.pytorch.org/whl/cpu
    !uv pip install -q --python {PY312} numpy scipy matplotlib "open3d==0.19.0" "evo==1.38.1" addict scikit-learn tqdm omegaconf PyYAML trimesh plyfile einops safetensors huggingface_hub opencv-python-headless pillow imageio "moviepy<2"
    !uv pip install -q --python {PY312} --no-deps -e .
    !{PY312} scripts/check_open3d.py --out /tmp/open3d_check_py312.json
    try:
        SPLIT = bool(json.load(open("/tmp/open3d_check_py312.json"))["legacy_tsdf_ok"])
    except Exception as e:
        print("Python 3.12 scoring environment not usable:", repr(e))
    if SPLIT:
        MODES, RECON_VARIANTS = "pose recon_unposed recon_posed", ""
        print("OK: the reconstruction scores (F1 / Chamfer) are computed with Python 3.12 + Open3D 0.19.0.")
    else:
        print("The Python 3.12 scoring environment does not work either: this run stays POSE ONLY. "
              "Send the output of this cell.")
SCORER = PY312 if SPLIT else "python"   # the Python that scores (and runs the analyses)

### Our metric code vs the released benchmark functions (should be ~0)

In [ ]:
!python scripts/check_metrics_vs_official.py --official-src {OFFICIAL_SRC} | tee /kaggle/working/metrics_crosscheck.json

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile
BENCH = "/tmp/da3bench"; os.makedirs(BENCH, exist_ok=True)
DATASETS = ["hiroom", "7scenes", "dtu64"]          # add "dtu", "scannetpp", "eth3d" if disk/time allow
for d in DATASETS:
    if not os.path.exists(f"{BENCH}/{d}"):
        z = hf_hub_download("depth-anything/DA3-BENCH", f"{d}.zip", repo_type="dataset", local_dir="/tmp/zips")
        zipfile.ZipFile(z).extractall(BENCH); os.remove(z)
!du -sh {BENCH}/*

In [ ]:
CKPT = ""   # e.g. "/kaggle/input/<02_train output>/runs/reduced_vits_tartanair_vkitti2/ckpt/last.pt"
MAX_SCENES = 0  # 0 = all scenes
for w, tag in [("official:da3-small", "official_da3-small"), ("official:da3-base", "official_da3-base")] + \
              ([(CKPT, "reduced_vits")] if CKPT else []):
    ds = " ".join(DATASETS)
    t = f"{tag}_{'-'.join(DATASETS)}" + (f"_limited{MAX_SCENES}" if MAX_SCENES else "")
    if SPLIT:   # the model here (GPU), the scores in the Python 3.12 environment (CPU)
        !python scripts/eval_bench.py --official-src {OFFICIAL_SRC} --bench-root {BENCH} \
            --work-root /tmp/eval --weights {w} --datasets {ds} \
            --modes {MODES} --max-scenes {MAX_SCENES} --tag {t} --infer-only
        !{PY312} scripts/eval_bench.py --official-src {OFFICIAL_SRC} --bench-root {BENCH} \
            --work-root /tmp/eval --weights {w} --datasets {ds} \
            --modes {MODES} --max-scenes {MAX_SCENES} --tag {t} --eval-only
    else:
        !python scripts/eval_bench.py --official-src {OFFICIAL_SRC} --bench-root {BENCH} \
            --work-root /tmp/eval --weights {w} --datasets {ds} \
            --modes {MODES} --max-scenes {MAX_SCENES} --tag {t}

In [ ]:
!python scripts/make_tables.py --out /kaggle/working/results_tables.md && cat /kaggle/working/results_tables.md